# MEME Sparklines — Interactive Hierarchy Viewer

Exposition notebook. All logic lives in the `sparklines_v2` package; this notebook is a thin driver that:

1. Chooses a time window.
2. Fetches the composite hierarchy from the archiver.
3. Renders the interactive `HierarchySparklineViewer`.

See `sparklines_v2.archive.hierarchy.build_composite_hierarchy` for the composite-PV math.

In [ ]:
%matplotlib widget

import datetime as dt
import os

import matplotlib.pyplot as plt

from sparklines_v2 import (
    HierarchySparklineViewer,
    build_composite_hierarchy,
    load_pv_groups,
    plot_percentile_band,
)
from sparklines_v2.plot import get_archive_data

# Bypass the site proxy for the archive appliance host.
os.environ["NO_PROXY"] = "lcls-archapp.slac.stanford.edu,.slac.stanford.edu"
os.environ["no_proxy"] = os.environ["NO_PROXY"]

In [ ]:
# Time window to analyze.
start = dt.datetime(2026, 3, 30, 22, 0, 0)
end = dt.datetime(2026, 3, 31, 6, 0, 0)

## GDET 241 percentile band

The pulse-energy monitor is best viewed as a rolling upper-tail percentile envelope.

In [ ]:
from sparklines_v2.archive import DEFAULT_MONITOR_SPECS

gdet_241 = DEFAULT_MONITOR_SPECS["GDET 241"]
data = get_archive_data(f"mean_1({gdet_241['pv_name']})", from_time=start, to_time=end)

ax, band_data = plot_percentile_band(
    data,
    window_size=10,
    value_scale=float(gdet_241.get("value_scale", 1.0)),
    y_label="pulse energy (µJ)",
)
plt.show()

## Composite hierarchy

Each PV group is folded to a single "composite PV" — the mean of per-PV normalized deviations from the interval baseline. See `sparklines_v2.archive.hierarchy._build_composite_from_series` for the math.

In [ ]:
pv_groups = load_pv_groups()
composite_hierarchy = build_composite_hierarchy(pv_groups, start, end)

print(f"Loaded {len(composite_hierarchy['pv_cache'])} PVs")
print(f"Built {sum(len(g['subgroups']) for g in composite_hierarchy['groups'].values())} subgroup composites")
print(f"Built {len(composite_hierarchy['groups'])} group composites")
print(f"Archive fetch wall time: {composite_hierarchy['timing']['fetch_wall_seconds']:.2f}s")
print(f"Hierarchy build wall time: {composite_hierarchy['timing']['build_wall_seconds']:.2f}s")
print(f"Skipped {len(composite_hierarchy['skipped_pvs'])} PVs with no usable archive payload")

In [ ]:
viewer = HierarchySparklineViewer(composite_hierarchy, start, end)
viewer.draw()